In [1]:
import pandas as pd

In [2]:
data = pd.read_json("s3://capstonedata2025healthcare/CapstoneHealthCare/Synthetic10K/syntheticmedicare10k/Aaron697_Shields502_c3df3ac0-d8a2-4e2c-9673-43f123441901.json")

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/fsspec/registry.py:298: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [3]:
data.head(10)

,resourceType,type,entry
0,Bundle,transaction,{'fullUrl': 'urn:uuid:4cb1a87c-6993-415f-aa95-...
1,Bundle,transaction,{'fullUrl': 'urn:uuid:eb3fc847-d368-3798-8ca0-...
2,Bundle,transaction,{'fullUrl': 'urn:uuid:0000016d-10a6-0f09-0000-...
3,Bundle,transaction,{'fullUrl': 'urn:uuid:9363553f-f9d8-4ecc-a1c6-...
4,Bundle,transaction,{'fullUrl': 'urn:uuid:1946f7c7-cdb2-453c-b0d4-...
5,Bundle,transaction,{'fullUrl': 'urn:uuid:ccdece26-bac6-42fb-8273-...
6,Bundle,transaction,{'fullUrl': 'urn:uuid:3354826a-a06a-37ba-860d-...
7,Bundle,transaction,{'fullUrl': 'urn:uuid:0000016d-10a6-0f09-0000-...
8,Bundle,transaction,{'fullUrl': 'urn:uuid:b7cbb06d-2ba4-4322-93bb-...
9,Bundle,transaction,{'fullUrl': 'urn:uuid:e1182b04-223e-42e7-b7e5-...


In [6]:
data['entry'].iloc[0].apply(lambda x: 

{'fullUrl': 'urn:uuid:4cb1a87c-6993-415f-aa95-871141dfb308',
 'resource': {'resourceType': 'Patient',
  'id': '4cb1a87c-6993-415f-aa95-871141dfb308',
  'text': {'status': 'generated',
   'div': '<div xmlns="http://www.w3.org/1999/xhtml">Generated by <a href="https://github.com/synthetichealth/synthea">Synthea</a>.Version identifier: v2.4.0-376-g05631c8e\n .   Person seed: 8350499292142924758  Population seed: 7373</div>'},
  'extension': [{'url': 'http://hl7.org/fhir/us/core/StructureDefinition/us-core-race',
    'extension': [{'url': 'ombCategory',
      'valueCoding': {'system': 'urn:oid:2.16.840.1.113883.6.238',
       'code': '2106-3',
       'display': 'White'}},
     {'url': 'text', 'valueString': 'White'}]},
   {'url': 'http://hl7.org/fhir/us/core/StructureDefinition/us-core-ethnicity',
    'extension': [{'url': 'ombCategory',
      'valueCoding': {'system': 'urn:oid:2.16.840.1.113883.6.238',
       'code': '2186-5',
       'display': 'Not Hispanic or Latino'}},
     {'url': 'te

In [7]:
import json, ast
from typing import Any, Dict, Optional

def parse_obj(obj_or_str: Any) -> Dict[str, Any]:
    """Accepts either a Python dict, a Python-literal string, or a JSON string."""
    if isinstance(obj_or_str, dict):
        return obj_or_str
    try:
        # first, try strict JSON
        return json.loads(obj_or_str)
    except Exception:
        # fall back to safe Python literal (handles single quotes, True/False, etc.)
        return ast.literal_eval(obj_or_str)

def first(items, default=None):
    for x in items or []:
        return x
    return default

def extract_patient_demographics(bundle_entry: Dict[str, Any]) -> Dict[str, Any]:
    r = bundle_entry.get("resource", {})
    # Name components
    name0 = first(r.get("name"))
    prefix = " ".join(name0.get("prefix", [])) if name0 else ""
    given = " ".join(name0.get("given", [])) if name0 else ""
    family = name0.get("family") if name0 else ""
    full_name = " ".join([p for p in [prefix, given, family] if p])

    # Phone/email (FHIR telecom array)
    phone = None
    email = None
    for t in r.get("telecom", []):
        if t.get("system") == "phone" and not phone:
            phone = t.get("value")
        if t.get("system") == "email" and not email:
            email = t.get("value")

    # Address (first)
    addr0 = first(r.get("address"))
    address_str = None
    if addr0:
        parts = []
        if addr0.get("line"): parts.append(", ".join(addr0["line"]))
        for k in ["city", "state", "postalCode", "country"]:
            if addr0.get(k):
                parts.append(addr0[k])
        address_str = ", ".join(parts)

    # Identifiers → map to a friendly type name
    identifiers = {}
    for ident in r.get("identifier", []):
        type_text: Optional[str] = None
        id_type = ident.get("type", {})
        # Prefer explicit text; else the first coding display/code; else system
        type_text = id_type.get("text")
        if not type_text:
            coding0 = first(id_type.get("coding"))
            if coding0:
                type_text = coding0.get("display") or coding0.get("code")
        key = type_text or ident.get("system") or "Unknown"
        identifiers[key] = ident.get("value")

    return {
        "resourceType": r.get("resourceType"),
        "resource_id": r.get("id"),
        "fullUrl": bundle_entry.get("fullUrl"),
        "name_full": full_name or None,
        "name_prefix": prefix or None,
        "name_given": given or None,
        "name_family": family or None,
        "gender": r.get("gender"),
        "birthDate": r.get("birthDate"),
        "phone": phone,
        "email": email,
        "address": address_str,
        "maritalStatus": (r.get("maritalStatus") or {}).get("text"),
        "identifiers": identifiers,  # dict like {"Medical Record Number": "...", "Social Security Number": "...", ...}
    }

# ---- run ----

In [9]:
obj = parse_obj(data['entry'].iloc[0])
out = extract_patient_demographics(obj)
from pprint import pprint
pprint(out)

{'address': '824 Moore Village Apt 18, Chandler, Arizona, US',
 'birthDate': '1953-02-19',
 'email': None,
 'fullUrl': 'urn:uuid:4cb1a87c-6993-415f-aa95-871141dfb308',
 'gender': 'male',
 'identifiers': {"Driver's License": 'S99927127',
                 'Medical Record Number': 'c3df3ac0-d8a2-4e2c-9673-43f123441901',
                 'Passport Number': 'X87771052X',
                 'Social Security Number': '999-53-9161',
                 'https://github.com/synthetichealth/synthea': 'c3df3ac0-d8a2-4e2c-9673-43f123441901'},
 'maritalStatus': 'M',
 'name_family': 'Shields502',
 'name_full': 'Mr. Aaron697 Shields502',
 'name_given': 'Aaron697',
 'name_prefix': 'Mr.',
 'phone': '555-398-7543',
 'resourceType': 'Patient',
 'resource_id': '4cb1a87c-6993-415f-aa95-871141dfb308'}


In [12]:
patientid= pd.DataFrame(out)
patientid

,resourceType,resource_id,fullUrl,name_full,name_prefix,name_given,name_family,gender,birthDate,phone,email,address,maritalStatus,identifiers
https://github.com/synthetichealth/synthea,Patient,4cb1a87c-6993-415f-aa95-871141dfb308,urn:uuid:4cb1a87c-6993-415f-aa95-871141dfb308,Mr. Aaron697 Shields502,Mr.,Aaron697,Shields502,male,1953-02-19,555-398-7543,None,"824 Moore Village Apt 18, Chandler, Arizona, US",M,c3df3ac0-d8a2-4e2c-9673-43f123441901
Medical Record Number,Patient,4cb1a87c-6993-415f-aa95-871141dfb308,urn:uuid:4cb1a87c-6993-415f-aa95-871141dfb308,Mr. Aaron697 Shields502,Mr.,Aaron697,Shields502,male,1953-02-19,555-398-7543,None,"824 Moore Village Apt 18, Chandler, Arizona, US",M,c3df3ac0-d8a2-4e2c-9673-43f123441901
Social Security Number,Patient,4cb1a87c-6993-415f-aa95-871141dfb308,urn:uuid:4cb1a87c-6993-415f-aa95-871141dfb308,Mr. Aaron697 Shields502,Mr.,Aaron697,Shields502,male,1953-02-19,555-398-7543,None,"824 Moore Village Apt 18, Chandler, Arizona, US",M,999-53-9161
Driver's License,Patient,4cb1a87c-6993-415f-aa95-871141dfb308,urn:uuid:4cb1a87c-6993-415f-aa95-871141dfb308,Mr. Aaron697 Shields502,Mr.,Aaron697,Shields502,male,1953-02-19,555-398-7543,None,"824 Moore Village Apt 18, Chandler, Arizona, US",M,S99927127
Passport Number,Patient,4cb1a87c-6993-415f-aa95-871141dfb308,urn:uuid:4cb1a87c-6993-415f-aa95-871141dfb308,Mr. Aaron697 Shields502,Mr.,Aaron697,Shields502,male,1953-02-19,555-398-7543,None,"824 Moore Village Apt 18, Chandler, Arizona, US",M,X87771052X


In [13]:
# pip install pandas boto3 pyarrow  (pyarrow if you save Parquet)
import boto3, pandas as pd, json, ast, gzip, io, sys
from urllib.parse import urlparse
from typing import Any, Dict, Iterable, List, Optional

# -------------------------
# Helpers
# -------------------------
def s3_list_keys(bucket: str, prefix: str) -> Iterable[str]:
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            # only JSON-like files; skip side files
            if key.endswith((".json", ".json.gz", ".jsonl", ".jsonl.gz")):
                yield key

def s3_read_bytes(bucket: str, key: str) -> bytes:
    s3 = boto3.client("s3")
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read()

def decode_text(data: bytes, encodings=("utf-8", "latin1")) -> str:
    for enc in encodings:
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            continue
    # last resort: replace errors
    return data.decode(encodings[0], errors="replace")

def parse_json_or_python_literal(text: str) -> Any:
    """Try strict JSON first; fall back to safe Python literal (handles single quotes)."""
    try:
        return json.loads(text)
    except Exception:
        return ast.literal_eval(text)

def first(iterable, default=None):
    for x in iterable or []:
        return x
    return default

def extract_patient_rows(obj: Any, source_key: str) -> List[Dict[str, Any]]:
    """
    Returns one row dict per Patient resource found.
    Works if the file is a Bundle with entry[*], or a single Patient resource.
    """
    rows = []

    def patient_to_row(r: Dict[str, Any]) -> Dict[str, Any]:
        name0 = first(r.get("name"))
        prefix = " ".join(name0.get("prefix", [])) if name0 else None
        given = " ".join(name0.get("given", [])) if name0 else None
        family = name0.get("family") if name0 else None
        full_name = " ".join([p for p in [prefix, given, family] if p]) or None

        phone, email = None, None
        for t in r.get("telecom", []) or []:
            sys_ = t.get("system")
            if sys_ == "phone" and phone is None:
                phone = t.get("value")
            if sys_ == "email" and email is None:
                email = t.get("value")

        addr0 = first(r.get("address"))
        address_str = None
        if addr0:
            parts = []
            if addr0.get("line"): parts.append(", ".join(addr0["line"]))
            for k in ["city", "state", "postalCode", "country"]:
                if addr0.get(k): parts.append(addr0[k])
            address_str = ", ".join(parts) if parts else None

        # identifiers into a flat dict of type -> value
        identifiers: Dict[str, Any] = {}
        for ident in r.get("identifier", []) or []:
            id_type_txt = (ident.get("type") or {}).get("text")
            if not id_type_txt:
                coding0 = first((ident.get("type") or {}).get("coding"))
                if coding0:
                    id_type_txt = coding0.get("display") or coding0.get("code")
            key = id_type_txt or ident.get("system") or "Identifier"
            identifiers[key] = ident.get("value")

        row = {
            "source_key": source_key,
            "resourceType": r.get("resourceType"),
            "resource_id": r.get("id"),
            "name_full": full_name,
            "name_prefix": prefix,
            "name_given": given,
            "name_family": family,
            "gender": r.get("gender"),
            "birthDate": r.get("birthDate"),
            "phone": phone,
            "email": email,
            "address": address_str,
            "maritalStatus": (r.get("maritalStatus") or {}).get("text"),
        }
        # merge identifiers (column names become id types)
        for k, v in identifiers.items():
            row[f"identifier::{k}"] = v
        return row

    # Case A: Bundle with entries
    if isinstance(obj, dict) and "entry" in obj:
        for e in obj.get("entry") or []:
            r = (e or {}).get("resource") or {}
            if r.get("resourceType") == "Patient":
                rows.append(patient_to_row(r))
        return rows

    # Case B: A single Patient resource
    if isinstance(obj, dict) and obj.get("resourceType") == "Patient":
        rows.append(patient_to_row(obj))
        return rows

    # Case C: The provided example shape (top-level object with "resource")
    if isinstance(obj, dict) and isinstance(obj.get("resource"), dict) and obj["resource"].get("resourceType") == "Patient":
        rows.append(patient_to_row(obj["resource"]))
        return rows

    return rows  # empty (no Patient found)

def load_one_file_to_dataframe(bucket: str, key: str) -> pd.DataFrame:
    raw = s3_read_bytes(bucket, key)
    if key.endswith(".gz"):
        raw = gzip.decompress(raw)
    text = decode_text(raw)
    # Some files might be JSON Lines (multiple objects per line)
    if "\n" in text and text.strip().split("\n")[0].strip().startswith("{") and not text.strip().startswith("{"):
        # NDJSON: build rows across lines
        rows_all: List[Dict[str, Any]] = []
        for line in text.splitlines():
            line = line.strip()
            if not line: 
                continue
            try:
                obj = parse_json_or_python_literal(line)
                rows_all.extend(extract_patient_rows(obj, key))
            except Exception:
                # skip malformed line
                pass
        return pd.DataFrame(rows_all)
    else:
        obj = parse_json_or_python_literal(text)
        rows = extract_patient_rows(obj, key)
        return pd.DataFrame(rows)

# -------------------------
# Main: build one DataFrame per file
# -------------------------
def build_per_file_dataframes(s3_prefix: str, limit: Optional[int] = None):
    """
    Returns a dict: {s3_key: pandas.DataFrame}
    Each DataFrame has one row per Patient found in that file.
    """
    u = urlparse(s3_prefix)
    bucket, prefix = u.netloc, u.path.lstrip("/")
    results: Dict[str, pd.DataFrame] = {}
    count = 0

    for key in s3_list_keys(bucket, prefix):
        try:
            df = load_one_file_to_dataframe(bucket, key)
            results[key] = df
        except Exception as e:
            print(f"[WARN] Skipping {key}: {e}", file=sys.stderr)
        count += 1
        if limit and count >= limit:
            break
    return results

# -------------------------
# Optional: save each DF back to S3
# -------------------------
def save_dataframes_to_s3(dfs_by_key: Dict[str, pd.DataFrame], dest_prefix: str, fmt: str = "parquet"):
    """
    Save each per-file DataFrame to S3 under dest_prefix, preserving filename.
    fmt: 'parquet' or 'csv'
    """
    u = urlparse(dest_prefix)
    bucket, prefix = u.netloc, u.path.lstrip("/")
    s3 = boto3.client("s3")

    for key, df in dfs_by_key.items():
        if df is None or df.empty:
            continue
        base = key.rsplit("/", 1)[-1].replace(".gz", "")
        base_noext = base.rsplit(".", 1)[0]
        if fmt == "parquet":
            buf = io.BytesIO()
            df.to_parquet(buf, index=False)
            outkey = f"{prefix.rstrip('/')}/{base_noext}.parquet"
            s3.put_object(Bucket=bucket, Key=outkey, Body=buf.getvalue())
        elif fmt == "csv":
            buf = io.StringIO()
            df.to_csv(buf, index=False)
            outkey = f"{prefix.rstrip('/')}/{base_noext}.csv"
            s3.put_object(Bucket=bucket, Key=outkey, Body=buf.getvalue().encode("utf-8"))
        else:
            raise ValueError("fmt must be 'parquet' or 'csv'")

In [ ]:
-------------------------
Example usage
-------------------------
s3_src = "s3://your-bucket/path/to/json/"
per_file = build_per_file_dataframes(s3_src)        # dict of DataFrames: {key: df}
print(per_file[next(iter(per_file))].head())        # peek first file's DF
save_dataframes_to_s3(per_file, "s3://your-bucket/output/patient_demographics/", fmt="parquet")